Main run: analysed primary pins + edge pins, both tasks, all three models.

Analysed = gold main sources minus `excluded_sources`, pins in `primary_bins`; edge = clicked ±1 m pins (secondary result).
OpenAI/Anthropic via batch (identical request bodies, 50% price), Google live. Images are read in slices to bound memory.
Only after the pilot passed and `renders/hashes.json` (run hash) is written. Cost ~$200-300.

In [8]:
import json

import duckdb
import pandas as pd

from thesis.config import REPO_ROOT, load_config
from thesis.vlm.run import configured_clients, run_dataset

out_dir = REPO_ROOT / "data" / "dataset_v2"
run_dir = out_dir / "main_run"
run_dir.mkdir(exist_ok=True)
SLICE = 500
scoring = load_config("scoring")

prompts = load_config("prompts")
TASK_PROMPTS = {
    "verification": next(p["text"] for p in prompts["prompts"] if p["id"] == "roleplaying"),
    "distance": next(p["text"] for p in prompts["distance_prompts"] if p["id"] == "roleplaying"),
}
clients = {client.provider: client for client in configured_clients()}

selected = pd.read_parquet(out_dir / "selection" / "sources.parquet")
pins = pd.read_parquet(out_dir / "audit" / "selected_pins.parquet")
gold = selected.merge(pd.read_json(out_dir / "gold" / "is_building.jsonl", lines=True), on="source_id")
excluded = {entry["source_id"] for entry in scoring["excluded_sources"]}
main_ids = gold.loc[
    gold["is_building"] & (gold["assignment"] == "main") & ~gold["source_id"].isin(excluded), "source_id"
]
edge_marks = pd.read_json(out_dir / "gold" / "edge_marks_phillip.jsonl", lines=True)
run_pins = pd.concat(
    [
        pins.loc[pins["source_id"].isin(main_ids) & pins["bin_id"].isin(scoring["primary_bins"])].assign(
            purpose="primary"
        ),
        pins.loc[
            pins["source_id"].isin(main_ids) & pins["pin_id"].isin(edge_marks.loc[~edge_marks["skipped"], "pin_id"])
        ].assign(purpose="edge"),
    ]
)
with duckdb.connect(str(out_dir / "manifest.duckdb"), read_only=True) as connection:
    renders = connection.execute("SELECT pin_id, image_path FROM renders WHERE preset_id = 'google_pin_red'").fetch_df()
run_pins = run_pins.merge(renders, on="pin_id").reset_index(drop=True)
assert run_pins["image_path"].notna().all()


def slices():
    for start in range(0, len(run_pins), SLICE):
        part = run_pins.iloc[start : start + SLICE]
        yield part, [(out_dir / path).read_bytes() for path in part["image_path"]]


hashes = json.loads((out_dir / "renders" / "hashes.json").read_text())
run_pins["purpose"].value_counts().to_dict(), hashes["run_hash"]

({'primary': 14205, 'edge': 193},
 'ebdf4b5e68da0975db2d21d5822b28a9dc400e20a53bfa90209abb7fd662d24f')

In [5]:
# submit: all OpenAI/Anthropic batches first, then Google live (blocks for a while).
# Safe to re-run: cached or already submitted requests are skipped, and new ids are added to batches.json.
batches_path = run_dir / "batches.json"
batch_ids = (
    json.loads(batches_path.read_text())
    if batches_path.exists()
    else {p: {t: [] for t in TASK_PROMPTS} for p in ("openai", "anthropic")}
)
for task, prompt in TASK_PROMPTS.items():
    for part, images in slices():
        items = [(image, prompt) for image in images]
        for provider in batch_ids:
            batch_ids[provider][task] += clients[provider].batch_classify(items, task=task)
        batches_path.write_text(json.dumps(batch_ids, indent=2))
print({provider: {task: len(ids) for task, ids in tasks.items()} for provider, tasks in batch_ids.items()})

for task, prompt in TASK_PROMPTS.items():
    for part, images in slices():
        run_dataset([(image, prompt) for image in images], task=task, clients=[clients["google"]])

{'openai': {'verification': 58, 'distance': 58}, 'anthropic': {'verification': 57, 'distance': 57}}


In [6]:
2 * len(run_pins)

28796

In [3]:
# re-run until nothing is None (None = still running)
batch_ids = json.loads((run_dir / "batches.json").read_text())
status = {
    (provider, task, batch_id): clients[provider].collect_batch(batch_id)
    for provider, tasks in batch_ids.items()
    for task, ids in tasks.items()
    for batch_id in ids
}
pd.Series(status).isna().sum(), sum(v or 0 for v in status.values())

(np.int64(0), 57591)

In [9]:
# read everything from the cache (requests a batch failed on are sent live here) and save the answers
rows = []
for task, prompt in TASK_PROMPTS.items():
    for part, images in slices():
        results = run_dataset([(image, prompt) for image in images], task=task, clients=list(clients.values()))
        for provider, responses in results.items():
            for pin, response in zip(part.itertuples(), responses):
                r = response.vlm_response
                rows.append(
                    {
                        "task": task,
                        "model": provider,
                        "purpose": pin.purpose,
                        "pin_id": pin.pin_id,
                        "source_id": pin.source_id,
                        "bin_id": pin.bin_id,
                        "exact_signed_distance_m": pin.exact_signed_distance_m,
                        "parsed_answer": r.parsed_answer,
                        "refusal_error_state": r.refusal_error_state,
                        "raw_text": r.raw_text,
                        "output_tokens": r.output_token_count,
                        "thinking_tokens": r.thinking_token_count,
                        "returned_model_id": r.returned_model_id,
                        "tier": r.price_schedule["tier"],
                        "cost": r.cost,
                        "timestamp": r.timestamp,
                        "run_hash": hashes["run_hash"],
                    }
                )
answers = pd.DataFrame(rows)
answers.to_parquet(run_dir / "answers.parquet")
answers.groupby(["task", "model"]).agg(
    n=("pin_id", "size"), invalid=("refusal_error_state", lambda s: s.notna().mean()), cost=("cost", "sum")
)

n   invalid       cost
task         model                                
distance     anthropic  14398  0.000417  41.016438
             google     14398  0.000000  51.810908
             openai     14398  0.000000  53.743172
verification anthropic  14398  0.012293  29.974457
             google     14398  0.000000  21.165459
             openai     14398  0.000000  17.756364